<a href="https://colab.research.google.com/github/NasorHidar/fly-rank-ml-1/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NasorHidar/fly-rank-ml-1/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import os

# Rule: Stay reproducible
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# 1. Load the dataset
url = "https://raw.githubusercontent.com/NasorHidar/fly-rank-ml-1/refs/heads/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 2. Prep features and target (matching Week 6)
df['needs_refresh'] = (df['trend_pct'] < 0).astype(int)
X = df.drop(columns=['content_id', 'client_id', 'trend_pct', 'trend_direction', 'needs_refresh'])
X = pd.get_dummies(X, drop_first=True)
y = df['needs_refresh']

# 3. Train final model on all available data for the playbook
model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
model.fit(X, y)

# 4. Generate the 'refresh_risk_score' (Probability of performance decay)
df['refresh_risk_score'] = model.predict_proba(X)[:, 1]

print("Model trained and data scored. Ready to build playbook.")

Model trained and data scored. Ready to build playbook.


## 1. Ranked actions + reason codes

The Queue Logic:

We prioritize actions based on a combination of the model's confidence (Risk Score) and the content's historical visibility.

Reason Codes:

* HIGH_VISIBILITY_DECAY: High model risk score (> 75%) AND above-median impressions. Action: Priority manual audit to prevent major traffic bleed.

* POOR_ENGAGEMENT_LEAK: High model risk score (> 75%) AND below-median CTR. Action: Standard refresh focusing on meta titles and search intent alignment.

* STANDARD_DECAY: Moderate risk score (60% - 75%). Action: Batch review for lightweight updates.

In [2]:
# Calculate medians for thresholds
median_imp = df['impressions_90d'].median()
median_ctr = df['ctr'].median()

def assign_reason(row):
    if row['refresh_risk_score'] > 0.75 and row['impressions_90d'] > median_imp:
        return 'HIGH_VISIBILITY_DECAY'
    elif row['refresh_risk_score'] > 0.75 and row['ctr'] < median_ctr:
        return 'POOR_ENGAGEMENT_LEAK'
    elif row['refresh_risk_score'] > 0.60:
        return 'STANDARD_DECAY'
    else:
        return 'NO_ACTION_NEEDED'

# Apply rules and build the queue
df['playbook_reason'] = df.apply(assign_reason, axis=1)

# Filter out healthy content and rank by risk score
playbook_queue = df[df['playbook_reason'] != 'NO_ACTION_NEEDED'].copy()
playbook_queue = playbook_queue.sort_values(by='refresh_risk_score', ascending=False)

print(f"Playbook Queue generated: {len(playbook_queue)} items flagged for review.")
print(playbook_queue[['content_id', 'refresh_risk_score', 'playbook_reason']].head())

Playbook Queue generated: 19715 items flagged for review.
                 content_id  refresh_risk_score        playbook_reason
29865  content_07f879e82acd                 1.0   POOR_ENGAGEMENT_LEAK
29904  content_52410ff246c6                 1.0  HIGH_VISIBILITY_DECAY
10676  content_fd502725c8dd                 1.0  HIGH_VISIBILITY_DECAY
10693  content_45d3aa65f4df                 1.0   POOR_ENGAGEMENT_LEAK
29939  content_c16e7735a15f                 1.0  HIGH_VISIBILITY_DECAY


## 2. Intended use and limits

Intended Use:

This tool provides directional decision-support for content strategy teams. By systematically scoring content decay risk, it reduces the manual effort of auditing large site portfolios and helps allocate editorial resources to the pages where updates will provide the most measured value.

Known Limits (Where it stops being valid):

* New Content: Pages less than 90 days old do not have enough historical data for the model to generate reliable directional signals.

* High-Volatility Events: The model does not inherently understand sudden market shifts, viral trends, or core algorithm updates. Scores generated during massive SERP volatility should be treated with lower confidence.

## 3. Human review + the no-go list

Human Review Checklist:

Before initiating a refresh, an editor must verify:

* Has the underlying search intent for this keyword shifted (e.g., from informational to transactional)?

* Is the traffic drop purely seasonal?

The No-Go List (Strictly Do Not Automate):

* Legal & Compliance Pages: Privacy policies, terms of service, and compliance documents must never be flagged for automated content refreshes.

* Core Brand Pages: Homepages, "About Us", and core product landing pages require executive or brand-manager oversight, not machine-driven changes.

* Direct Title Overwrites: The model flags the need for an update, but must not automatically overwrite meta descriptions or titles directly to the CMS without human approval.

## 4. Monitoring / retrain triggers

Monitoring Strategy:
To ensure the model remains accurate, we must monitor the distribution of risk scores and the underlying data pipelines.

Retrain Triggers:

* Data Drift: If the median CTR or average impression volume shifts by more than 15% across the global dataset (e.g., due to Google introducing new AI overview snippets), the model's thresholds must be recalibrated.

* Performance Degradation: If the validation ROC-AUC drops below our measured baseline of 0.70 during quarterly audits, the model must be retrained on the most recent 6 months of data.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
# Prepare the final dataframe for export
export_cols = [
    'content_id', 'client_id', 'content_type', 'impressions_90d', 'ctr',
    'refresh_risk_score', 'playbook_reason'
]
final_export = playbook_queue[export_cols]

# Ensure the output directory exists
os.makedirs('../work/outputs/', exist_ok=True)

# Export the queue to CSV for the capstone paper
export_path = '../work/outputs/action_playbook_queue.csv'
final_export.to_csv(export_path, index=False)

print(f"Action Playbook successfully exported to: {export_path}")

Action Playbook successfully exported to: ../work/outputs/action_playbook_queue.csv


### Self-check

Before you submit, confirm each line honestly:

*   [x] Every section above is filled — markdown thinking AND the code that backs it
*   [x] The notebook runs top to bottom with no errors (Runtime → Run all)
*   [x] No client names, URLs, or private queries anywhere
*   [x] My claims use careful words: observed, measured, directional, decision-support
*   [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.